> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 00 · FOUNDATION</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Setup and the ₹ Cost Meter</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Everything else depends on this notebook. Run it once, keep the meter.</div>
</div>

**Time:** 15 min &nbsp;·&nbsp; **Est. cost:** ₹0.02 &nbsp;·&nbsp; **Prereq:** Python 3.11+, a Sarvam API key

## What you build here

1. A working SDK connection
2. **A cost meter** — the `₹` tally every other lab imports
3. Your first deliberate failure (`content is None`) and its two fixes

> **Teaching note.** Do not skip the cost meter. The habit of printing rupees after
> every call is the single thing that separates this course from an API tour.

### 1 · Install

In [1]:
# Run once. Restart the kernel afterwards if the import fails.
%pip install -q sarvamai python-dotenv requests websockets

Note: you may need to restart the kernel to use updated packages.


### 2 · Your key

Get one free at **dashboard.sarvam.ai** — new accounts include ₹100 of credit,
which is more than enough for every lab in this series.

Create a file called `.env` next to this notebook:

```
SARVAM_API_KEY=sk_xxxxxxxxxxxxxxxx
```

In [2]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path



from dotenv import load_dotenv, find_dotenv

# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)



In [3]:
# pip install sarvamai python-dotenv
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)

DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


### 3 · Smoke test — is the key alive?

In [4]:
r = client.chat.completions(
    model="sarvam-105b",
    messages= [{"role": "user", "content": "एक वाक्य में बताइए: भारत में UPI क्यों सफल हुआ?"}],
    max_tokens=3000,          # note: generous. See section 5 for why.
)
print(r.choices[0].message.content)

यूपीआई (UPI) कम खर्च वाले, असली समय के डिजिटल पेमेंट के लिए एक लोकप्रिय और सबके काम आने वाली स्टैंडर्ड बनाने में सफल रहा। इसने भारत के तेजी से बढ़ते मोबाइल बाजार का फायदा उठाया और सरकार के डिजिटल इंफ्रास्ट्रक्चर प्रोग्राम्स के साथ मिलकर बहुत तेज़ी से लोगों ने इसे अपनाया।


In [5]:
print(r)

id='20260828_fed106cb-81e0-4708-910a-ff5e848e8aaa' choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionResponseMessage(content='यूपीआई (UPI) कम खर्च वाले, असली समय के डिजिटल पेमेंट के लिए एक लोकप्रिय और सबके काम आने वाली स्टैंडर्ड बनाने में सफल रहा। इसने भारत के तेजी से बढ़ते मोबाइल बाजार का फायदा उठाया और सरकार के डिजिटल इंफ्रास्ट्रक्चर प्रोग्राम्स के साथ मिलकर बहुत तेज़ी से लोगों ने इसे अपनाया।', refusal=None, reasoning_content='1.  **Deconstruct the User\'s Request:**\n    *   This Hindi query asks to state in one sentence why UPI succeeded (एक वाक्य में बताइए) in India (भारत में).\n    *   **Core Task:** Explain the success of UPI in India.\n    *   **Constraint:** "in one sentence" (एक वाक्य में).\n    *   **Context:** India. This is crucial. The answer must be India-specific.\n\n2.  **Brainstorm the Key Success Factors of UPI in India:**\n    *   **Government Push:** National Payments Corporation of India (NPCI) is a key player. The government\'s "

### 4 · The cost meter

Every billing unit on the platform, wrapped in one class. Import it into every
later notebook, or just re-run this cell.

In [6]:
# ── The ₹ meter. Every lab uses this. ──────────────────────────────────────
# Rates as of August 2026. Verify at docs.sarvam.ai/api/getting-started/pricing
# The calculations are imitations and not exact readings from the usage bill from Sarvam.  

FREE_CREDIT = 1000.00          # new Sarvam accounts
RATES = {
    "stt_per_hour":            30.00,
    "stt_diarized_per_hour":   45.00,
    "translate_per_10k":       20.00,
    "transliterate_per_10k":   20.00,
    "lid_per_10k":              3.50,
    "tts_v2_per_10k":          15.00,
    "tts_v3_per_10k":          30.00,
    "llm_in_per_1m":           29.28,
    "llm_cached_in_per_1m":    10.98,
    "llm_out_per_1m":          73.20,
    "doc_per_page":             0.50,
    "samvaad_per_min":          3.50,
}

class CostMeter: 
    """Running ₹ tally for Sarvam API calls, calculated from published rates
    as of August 2026 (see RATES dict). Estimate only — actual invoice may
    differ due to rate changes, rounding, prompt caching and free-tier
    consumption. In fact, the meter is a conservative upper bound on cost 
    in almost every real scenario if rate changes are not applied. Verify 
    current pricing at docs.sarvam.ai/api/getting-started/pricing.
    """
    def __init__(self): self.items = []

    def add(self, label, rupees, detail=""):
        self.items.append({"label": label, "inr": rupees, "detail": detail})
        return rupees

    # --- convenience wrappers, one per billing unit -----------------------
    def stt(self, seconds, diarized=False):
        r = RATES["stt_diarized_per_hour" if diarized else "stt_per_hour"] / 3600 * seconds
        return self.add("STT" + (" +diar" if diarized else ""), r, f"{seconds:.1f}s")

    def tts(self, chars, v3=True):
        r = RATES["tts_v3_per_10k" if v3 else "tts_v2_per_10k"] / 10_000 * chars
        return self.add(f"TTS {'v3' if v3 else 'v2'}", r, f"{chars} chars")

    def text(self, chars, kind="translate"):
        r = RATES[f"{kind}_per_10k"] / 10_000 * chars
        return self.add(kind, r, f"{chars} chars")

    def llm(self, in_tok, out_tok, cached_tok=0):
        r = ((in_tok - cached_tok) * RATES["llm_in_per_1m"]
             + cached_tok * RATES["llm_cached_in_per_1m"]
             + out_tok * RATES["llm_out_per_1m"]) / 1_000_000
        return self.add("LLM", r, f"{in_tok} in / {out_tok} out / {cached_tok} cached")

    def doc(self, pages):
        return self.add("DocAI", RATES["doc_per_page"] * pages, f"{pages} pages")

    def report(self):
        if not self.items:
            print("nothing billed yet"); return 0.0
        w = max(len(i["label"]) for i in self.items) + 2
        print("─" * (w + 34))
        for i in self.items:
            print(f"{i['label']:<{w}} ₹{i['inr']:>9.4f}   {i['detail']}")
        total = sum(i["inr"] for i in self.items)
        print("─" * (w + 34))
        print(f"{'TOTAL':<{w}} ₹{total:>9.4f}")
        print(f"{'':<{w}}  (₹{FREE_CREDIT:.0f} free credit → ₹{FREE_CREDIT-total:.2f} left)")
        return total

cost = CostMeter()
print("cost meter armed")

cost meter armed


In [7]:
# Try it — the smoke test above, costed
u = r.usage
cost.llm(u.prompt_tokens, u.completion_tokens)
cost.report()

───────────────────────────────────────
LLM   ₹   0.1103   22 in / 1498 out / 0 cached
───────────────────────────────────────
TOTAL ₹   0.1103
       (₹1000 free credit → ₹999.89 left)


0.11029776000000001

### 5 · Your first deliberate failure

`reasoning_effort` defaults to `"low"` on Sarvam-105B, **and reasoning tokens count
against `max_tokens`.** With a small budget the model spends it all thinking and
returns `content = None`.

This is the #1 support question on this platform. Cause it on purpose now so you
recognise it instantly later.

In [8]:
# ⚠️ THIS IS SUPPOSED TO FAIL
bad = client.chat.completions(
    model="sarvam-105b",
    messages=[{"role": "user", "content": "Explain GST in three sentences."}],
    max_tokens=60,            # too small — reasoning eats it
)
print("content :", bad.choices[0].message.content)
print("tokens  :", bad.usage.completion_tokens, "completion tokens billed anyway")

content : None
tokens  : 60 completion tokens billed anyway


In [9]:
# FIX A — give it room to think AND answer
ok_a = client.chat.completions(
    model="sarvam-105b",
    messages=[{"role": "user", "content": "Explain GST in three sentences."}],
    max_tokens=2000,
)
print("FIX A:", ok_a.choices[0].message.content[:200], "...\n")


FIX A: The Goods and Services Tax (GST) is a comprehensive, destination-based tax on the supply of goods and services in India, replacing a complex structure of multiple previous indirect taxes. Launched on  ...



In [10]:
# FIX B — turn reasoning off entirely (faster, cheaper, fine for extraction)
ok_b = client.chat.completions(
    model="sarvam-105b",
    messages=[{"role": "user", "content": "Explain GST in three sentences."}],
    max_tokens=300,
    reasoning_effort=None,
)
print("FIX B:", ok_b.choices[0].message.content[:200])

FIX B: 
The Goods and Services Tax (GST) is a comprehensive indirect tax levied on the supply of goods and services across India, replacing a complex web of previous central and state taxes. Under this syste


| | Fix A — raise `max_tokens` | Fix B — `reasoning_effort=None` |
|---|---|---|
| Keeps reasoning | ✅ | ❌ |
| Cheaper | ❌ | ✅ |
| Faster | ❌ | ✅ |
| Use for | multi-step logic, agents | classification, extraction, routing |

**Plan ceilings on `max_tokens`:** Starter 4096 · Pro 16384 · Business 128000.

### 6 · Save the meter for reuse

In [11]:
Path("cost_meter.py").write_text('''
FREE_CREDIT = 1000.00

RATES = ''' + repr(RATES) + '''

class CostMeter:
    def __init__(self): self.items = []
    def add(self, label, rupees, detail=""):
        self.items.append({"label": label, "inr": rupees, "detail": detail}); return rupees
    def stt(self, seconds, diarized=False):
        r = RATES["stt_diarized_per_hour" if diarized else "stt_per_hour"]/3600*seconds
        return self.add("STT", r, f"{seconds:.1f}s")
    def tts(self, chars, v3=True):
        r = RATES["tts_v3_per_10k" if v3 else "tts_v2_per_10k"]/10_000*chars
        return self.add("TTS", r, f"{chars} chars")
    def text(self, chars, kind="translate"):
        return self.add(kind, RATES[f"{kind}_per_10k"]/10_000*chars, f"{chars} chars")
    def llm(self, i, o, c=0):
        r = ((i-c)*RATES["llm_in_per_1m"] + c*RATES["llm_cached_in_per_1m"]
             + o*RATES["llm_out_per_1m"])/1_000_000
        return self.add("LLM", r, f"{i} in / {o} out")
    def doc(self, pages): return self.add("DocAI", RATES["doc_per_page"]*pages, f"{pages} pages")
    def report(self):
        for i in self.items: print(f"{i['label']:<12} ₹{i['inr']:>9.4f}  {i['detail']}")
        t = sum(i["inr"] for i in self.items)
        print(f"{'TOTAL':<12} ₹{t:>9.4f}")
        print(f"{'':<12}  (₹{FREE_CREDIT:.0f} free credit → ₹{FREE_CREDIT-t:.2f} left)")
        print(f"{'':<12}  Estimated from published rates; actual billing usually lower")
        return t
''')
print("wrote cost_meter.py — later labs do:  from cost_meter import CostMeter")

wrote cost_meter.py — later labs do:  from cost_meter import CostMeter


---
## ✅ Checkpoint

- [ ] `client` connects and returns Hindi text
- [ ] You have caused `content is None` and fixed it two ways
- [ ] `cost.report()` prints a rupee total
- [ ] `cost_meter.py` exists on disk

## 🧪 Try this

1. Set `max_tokens=200` with `reasoning_effort=None`. Does it work now? Why?
2. Run the same prompt at `temperature=0` twice with `seed=42`. Identical?
3. Swap `sarvam-105b` → `sarvam-30b`. Compare latency, cost and answer quality.